# Chapter 9 §9.3: Marketing Blog Writer with Style Transfer

This notebook accompanies Chapter 9 §9.3 of *Context Engineering with DSPy*: **Marketing Blog Writer with Style Transfer**.

**Required environment variables (set in your `.env`):**

- `OPENAI_API_KEY`

## Overview

```
9.3 Marketing Blog Writer with Style Transfer

Demonstrates three patterns for creative content generation:
1. Style profile generation — extract a brand's writing style from examples
2. Style transfer — use the profile to constrain generation
3. Semantic distance — evaluate style consistency with embeddings

The challenge: creative output has no ground truth. You can't use exact match.
BLEU and ROUGE measure surface similarity, not whether the writing sounds like
your brand. The answer is embedding-based semantic distance.

Failure mode: Context Distraction (Ch 1.4.3) — without style grounding, the
    model defaults to generic AI prose
Technique: FewShot Learning + Prompt Optimization
Agentic pattern: Prompt Chaining (generate style -> transfer -> evaluate)

Usage:
    python blog_writer.py                          # full demo
    python blog_writer.py --topic "New API launch"  # custom topic
    python blog_writer.py --style technical         # specific style
```


In [ ]:
%pip install -r ../requirements.txt -q

## Imports and LM configuration

In [ ]:
import os
from dotenv import load_dotenv

import numpy as np
import dspy
from dspy.evaluate import Evaluate

load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")
lm = dspy.LM("openai/gpt-4o-mini", api_key=api_key)
embedder = dspy.Embedder("openai/text-embedding-3-small", api_key=api_key)
dspy.configure(lm=lm, embedder=embedder)

## Style definitions and reference examples

In [ ]:
STYLES = {
    "technical": {
        "description": "Technical, precise, data-driven. Use specific metrics and benchmarks. Short sentences. Avoid hype. Write for engineers.",
        "references": [
            "Performance benchmarks show 99.9% uptime across all regions. Latency: p50=12ms, p99=45ms. Zero data loss during the Q3 migration.",
            "The new query planner reduces cold-start time by 40%. Memory footprint dropped from 2.1GB to 890MB. Tested across 50k concurrent connections.",
            "Our CDN edge cache hit rate sits at 94%. Cache invalidation propagates in under 200ms. Origin shield reduces backend load by 6x.",
        ],
    },
    "conversational": {
        "description": "Friendly, approachable, casual. Ask questions. Use contractions and first person. Feel like a smart friend explaining something.",
        "references": [
            "Want to ship faster? Here's the thing — your deploy pipeline doesn't need to be complicated. We built ours in an afternoon.",
            "You've got better things to do than babysit your CI/CD. Let's fix that. One config file, zero drama, and you're deploying on every push.",
            "I used to dread Mondays because of deploy anxiety. Now? I push to prod before my coffee gets cold. Here's how we got there.",
        ],
    },
    "executive": {
        "description": "Strategic, high-level, business-focused. Emphasize ROI and market position. Professional but accessible. Write for leadership.",
        "references": [
            "Strategic investments in platform reliability drove 40% revenue growth year-over-year. Customer retention improved by 12 points.",
            "Market analysis indicates growing demand for compliance-first infrastructure. Our SOC 2 and ISO 27001 certifications position us ahead of competitors.",
            "The enterprise segment represented 65% of new ARR this quarter. Average contract value increased 28% as customers consolidated vendors.",
        ],
    },
}

## Signatures

In [ ]:
class ExtractStyleProfile(dspy.Signature):
    """Analyze writing samples and extract a detailed style profile.

    Focus on: sentence length patterns, vocabulary choices, tone markers,
    use of data/metrics, rhetorical devices, and what makes the writing
    distinctive from generic AI-generated content.
    """

    writing_samples: str = dspy.InputField(desc="3-5 example paragraphs in the target style")
    style_profile: str = dspy.OutputField(
        desc="Detailed style profile covering: sentence structure, vocabulary, "
             "tone, use of data, rhetorical patterns, and what to avoid"
    )

class StyleAwareBlogWriter(dspy.Signature):
    """Write marketing content that matches a specific brand voice.

    The style profile constrains your writing. Do NOT default to generic
    AI prose — match the patterns described in the profile.
    """

    topic: str = dspy.InputField(desc="Blog topic or headline")
    style_profile: str = dspy.InputField(desc="Detailed style profile to match")
    content: str = dspy.OutputField(desc="Blog paragraph (3-5 sentences) matching the style")

class StyleTransfer(dspy.Signature):
    """Rewrite content in a different style while preserving all factual claims.

    Do not add or remove information. Only change how the information
    is presented: sentence structure, vocabulary, tone, and formatting.
    """

    original_content: str = dspy.InputField(desc="Content to transform")
    target_profile: str = dspy.InputField(desc="Target style profile")
    transformed_content: str = dspy.OutputField(desc="Same content rewritten in target style")

## Style-aware module

In [ ]:
class StyledBlogWriter(dspy.Module):
    """Two-stage blog writer: extract style profile, then generate content."""

    def __init__(self):
        super().__init__()
        self.extract_style = dspy.Predict(ExtractStyleProfile)
        self.write = dspy.ChainOfThought(StyleAwareBlogWriter)

    def forward(self, topic, writing_samples):
        # Stage 1: Extract style profile from examples
        profile = self.extract_style(writing_samples=writing_samples)

        # Stage 2: Generate content constrained by the profile
        result = self.write(topic=topic, style_profile=profile.style_profile)

        return dspy.Prediction(
            content=result.content,
            style_profile=profile.style_profile,
        )

## Semantic distance metric

In [ ]:
def cosine_similarity(a, b):
    """Cosine similarity between two vectors."""
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

def style_distance(content, reference_texts, embedder):
    """Average cosine similarity between content and reference style examples.

    Higher = more similar to the target style.
    This is the metric that tells you whether the output actually sounds
    like your brand, not just whether it's grammatically correct.
    """
    content_emb = embedder([content])[0]
    ref_embs = embedder(reference_texts)

    similarities = [cosine_similarity(content_emb, ref) for ref in ref_embs]
    return sum(similarities) / len(similarities)

def make_style_metric(style_name, embedder):
    """Create a metric function for a specific target style.

    Returns similarity to the target style. Higher is better.
    """
    refs = STYLES[style_name]["references"]

    def metric(example, pred, trace=None):
        return style_distance(pred.content, refs, embedder)

    return metric

## Demos

In [ ]:
def demo_style_profiles(embedder):
    """Show style profile extraction and generation across all three styles."""
    print("=" * 60)
    print("STYLE PROFILE EXTRACTION + GENERATION")
    print("=" * 60)

    writer = StyledBlogWriter()
    topic = "Our platform now supports 10,000 API requests per second"

    for style_name, style_data in STYLES.items():
        samples = "\n\n".join(style_data["references"])
        result = writer(topic=topic, writing_samples=samples)

        # Measure similarity to each style
        scores = {}
        for check_style, check_data in STYLES.items():
            scores[check_style] = style_distance(
                result.content, check_data["references"], embedder
            )

        best_match = max(scores, key=scores.get)
        hit = "MATCH" if best_match == style_name else "MISS"

        print(f"\n--- {style_name.upper()} ---")
        print(f"Generated: {result.content[:200]}...")
        print(f"Similarity: {', '.join(f'{k}={v:.3f}' for k, v in scores.items())}")
        print(f"Best match: {best_match} [{hit}]")

def demo_style_transfer(embedder):
    """Show content being transferred between styles."""
    print("\n" + "=" * 60)
    print("STYLE TRANSFER")
    print("=" * 60)

    transfer = dspy.ChainOfThought(StyleTransfer)
    profile_extractor = dspy.Predict(ExtractStyleProfile)

    # Start with technical content
    original = (
        "The v2 API endpoint processes requests in 12ms p50. "
        "Rate limiting is set at 10,000 req/min per org. "
        "Authentication uses OAuth 2.0 bearer tokens with 1-hour expiry."
    )

    print(f"\nOriginal (technical): {original}")

    for target_style in ["conversational", "executive"]:
        # Extract target profile
        samples = "\n\n".join(STYLES[target_style]["references"])
        profile = profile_extractor(writing_samples=samples)

        # Transfer
        result = transfer(
            original_content=original,
            target_profile=profile.style_profile,
        )

        score = style_distance(
            result.transformed_content,
            STYLES[target_style]["references"],
            embedder,
        )

        print(f"\n-> {target_style.upper()} (similarity: {score:.3f}):")
        print(f"   {result.transformed_content[:200]}...")

def demo_optimization(embedder):
    """Show optimization for style consistency."""
    print("\n" + "=" * 60)
    print("OPTIMIZATION FOR STYLE CONSISTENCY")
    print("=" * 60)

    # Training data: topics with their target style
    training_data = []
    for style_name, style_data in STYLES.items():
        samples = "\n\n".join(style_data["references"])
        training_data.append(
            dspy.Example(
                topic="New security features with end-to-end encryption",
                writing_samples=samples,
                _style=style_name,  # metadata for metric
            ).with_inputs("topic", "writing_samples")
        )
        training_data.append(
            dspy.Example(
                topic="Customer success story: 3x faster deployments",
                writing_samples=samples,
                _style=style_name,
            ).with_inputs("topic", "writing_samples")
        )

    # Metric: similarity to the correct style
    def multi_style_metric(example, pred, trace=None):
        target = getattr(example, "_style", "technical")
        refs = STYLES[target]["references"]
        return style_distance(pred.content, refs, embedder)

    # Baseline
    baseline = StyledBlogWriter()
    evaluator = Evaluate(devset=training_data, metric=multi_style_metric, display_progress=False)
    baseline_score = float(evaluator(baseline))

    # Optimize
    from dspy.teleprompt import BootstrapFewShot

    optimizer = BootstrapFewShot(
        metric=multi_style_metric,
        max_bootstrapped_demos=2,
        max_labeled_demos=1,
    )
    optimized = optimizer.compile(StyledBlogWriter(), trainset=training_data)
    optimized_score = float(evaluator(optimized))

    print(f"\n  Baseline style consistency:  {baseline_score:.1f}%")
    print(f"  Optimized style consistency: {optimized_score:.1f}%")
    print(f"  Delta:                       {optimized_score - baseline_score:+.1f}%")

## Run the demo

The code below mirrors the `main()` block in the source script with the argparse CLI branches removed — it runs the full demo path end-to-end.

In [ ]:
# Full demo
demo_style_profiles(embedder)
demo_style_transfer(embedder)

if not False:
    demo_optimization(embedder)